In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
movies_data = pd.read_csv('movies.csv')
ratings_data = pd.read_csv('ratings.csv')
links_data = pd.read_csv('links.csv')
tags_data = pd.read_csv('tags.csv')

In [4]:
movies_data.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [5]:
ratings_data.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


Analysis done in precheck.ipynb file

Conerting Unix timestamp measured in seconds to datetime

In [6]:
import pandas as pd

ratings = ratings_data.copy()

ratings["rating_date"] = pd.to_datetime(
    ratings["timestamp"],
    unit="s",
    utc=True
)

ratings = ratings.sort_values(
    ["timestamp", "userId", "movieId"]
).reset_index(drop=True)

ratings.head()

,userId,movieId,rating,timestamp,rating_date
0,429,22,4.0,828124615,1996-03-29 18:36:55+00:00
1,429,150,5.0,828124615,1996-03-29 18:36:55+00:00
2,429,161,5.0,828124615,1996-03-29 18:36:55+00:00
3,429,165,4.0,828124615,1996-03-29 18:36:55+00:00
4,429,218,4.0,828124615,1996-03-29 18:36:55+00:00


In [7]:
# Your previous cell already created and sorted "ratings".
ratings = ratings.sort_values(
    ["timestamp", "userId", "movieId"]
).reset_index(drop=True)

train_position = int(len(ratings) * 0.80)
validation_position = int(len(ratings) * 0.90)

train_cutoff = ratings["timestamp"].iloc[train_position]
validation_cutoff = ratings["timestamp"].iloc[validation_position]

if train_cutoff >= validation_cutoff:
    raise ValueError(
        "The time boundaries overlap. Choose different cutoffs."
    )

print(
    "Validation starts:",
    pd.to_datetime(train_cutoff, unit="s", utc=True)
)

print(
    "Test starts:",
    pd.to_datetime(validation_cutoff, unit="s", utc=True)
)

Validation starts: 2016-03-22 08:26:11+00:00
Test starts: 2017-06-21 07:17:30+00:00


In [8]:
train_ratings = ratings[
    ratings["timestamp"] < train_cutoff
].copy()

validation_ratings = ratings[
    (ratings["timestamp"] >= train_cutoff)
    & (ratings["timestamp"] < validation_cutoff)
].copy()

test_ratings = ratings[
    ratings["timestamp"] >= validation_cutoff
].copy()

print("Training ratings:", len(train_ratings))
print("Validation ratings:", len(validation_ratings))
print("Test ratings:", len(test_ratings))

Training ratings: 80668
Validation ratings: 10084
Test ratings: 10084


In [9]:
split_summary = pd.DataFrame([
    {
        "split": name,
        "ratings": len(data),
        "users": data["userId"].nunique(),
        "movies": data["movieId"].nunique(),
        "first_rating": data["rating_date"].min(),
        "last_rating": data["rating_date"].max()
    }
    for name, data in [
        ("Train", train_ratings),
        ("Validation", validation_ratings),
        ("Test", test_ratings)
    ]
])

display(split_summary)

assert (
    len(train_ratings) + len(validation_ratings) + len(test_ratings)
    == len(ratings)
)

assert (
    train_ratings["timestamp"].max()
    < validation_ratings["timestamp"].min()
)

assert (
    validation_ratings["timestamp"].max()
    < test_ratings["timestamp"].min()
)

print("Split checks passed.")

,split,ratings,users,movies,first_rating,last_rating
0,Train,80668,522,7867,1996-03-29 18:36:55+00:00,2016-03-22 08:26:02+00:00
1,Validation,10084,62,3261,2016-03-22 08:26:11+00:00,2017-06-21 07:17:25+00:00
2,Test,10084,74,4428,2017-06-21 07:17:30+00:00,2018-09-24 14:27:30+00:00


Split checks passed.


In [10]:
train_users = set(train_ratings["userId"])
train_movies = set(train_ratings["movieId"])

cold_start_rows = []

for name, df in {
    "Validation": validation_ratings,
    "Test": test_ratings
}.items():

    new_user = ~df["userId"].isin(train_users)
    new_movie = ~df["movieId"].isin(train_movies)

    warm_interaction = (~new_user) & (~new_movie)

    cold_start_rows.append({
        "split": name,
        "total_interactions": len(df),
        "new_users": df.loc[
            new_user, "userId"
        ].nunique(),
        "new_movies": df.loc[
            new_movie, "movieId"
        ].nunique(),
        "new_user_interactions": int(new_user.sum()),
        "new_movie_interactions": int(new_movie.sum()),
        "either_new_interactions": int(
            (new_user | new_movie).sum()
        ),
        "warm_interactions": int(
            warm_interaction.sum()
        ),
        "warm_percentage": round(
            100 * warm_interaction.mean(), 2
        )
    })

cold_start_summary = pd.DataFrame(cold_start_rows)

display(cold_start_summary)

,split,total_interactions,new_users,new_movies,new_user_interactions,new_movie_interactions,either_new_interactions,warm_interactions,warm_percentage
0,Validation,10084,41,719,9093,1015,9374,710,7.04
1,Test,10084,54,1347,9154,2029,9483,601,5.96


In [11]:
train_users = set(train_ratings["userId"])
train_movies = set(train_ratings["movieId"])

cold_start_results = []

for name, data in [
    ("Validation", validation_ratings),
    ("Test", test_ratings)
]:
    new_user = ~data["userId"].isin(train_users)
    new_movie = ~data["movieId"].isin(train_movies)
    warm = ~new_user & ~new_movie

    cold_start_results.append({
        "split": name,
        "users_seen_in_train": data.loc[
            ~new_user, "userId"
        ].nunique(),
        "users_not_in_train": data.loc[
            new_user, "userId"
        ].nunique(),
        "movies_not_in_train": data.loc[
            new_movie, "movieId"
        ].nunique(),
        "total_ratings": len(data),
        "warm_ratings": int(warm.sum()),
        "cold_ratings": int((new_user | new_movie).sum()),
        "warm_percentage": round(100 * warm.mean(), 2)
    })

cold_start_summary = pd.DataFrame(cold_start_results)
display(cold_start_summary)

,split,users_seen_in_train,users_not_in_train,movies_not_in_train,total_ratings,warm_ratings,cold_ratings,warm_percentage
0,Validation,21,41,719,10084,710,9374,7.04
1,Test,20,54,1347,10084,601,9483,5.96


In [12]:
popular_movies = (
    train_ratings.groupby("movieId")
    .agg(
        rating_count=("rating", "size"),
        average_rating=("rating", "mean")
    )
    .reset_index()
)

popular_movies = popular_movies.merge(
    movies_data,
    on="movieId",
    how="left",
    validate="one_to_one"
)

popular_movies = popular_movies.sort_values(
    ["rating_count", "movieId"],
    ascending=[False, True]
).reset_index(drop=True)

display(
    popular_movies[
        ["movieId", "title", "genres", "rating_count", "average_rating"]
    ].head(10)
)

,movieId,title,genres,rating_count,average_rating
0,356,Forrest Gump (1994),Comedy|Drama|Romance|War,267,4.166667
1,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,264,4.189394
2,318,"Shawshank Redemption, The (1994)",Crime|Drama,256,4.423828
3,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,235,4.202128
4,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,219,4.214612
5,480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,218,3.786697
6,110,Braveheart (1995),Action|Drama|War,212,4.058962
7,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,206,4.264563
8,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi,200,3.970000
9,527,Schindler's List (1993),Drama|War,188,4.228723


In [13]:
def recommend_popular(seen_movie_ids=None, n=10):
    if seen_movie_ids is None:
        seen_movie_ids = []

    recommendations = popular_movies[
        ~popular_movies["movieId"].isin(seen_movie_ids)
    ]

    return recommendations[
        ["movieId", "title", "genres", "rating_count", "average_rating"]
    ].head(n).reset_index(drop=True)

In [14]:
seen_movies = [356, 296, 318]

recommendations = recommend_popular(
    seen_movie_ids=seen_movies,
    n=10
)

display(recommendations)

assert not recommendations["movieId"].isin(seen_movies).any()
print("Already-rated movies were excluded.")

,movieId,title,genres,rating_count,average_rating
0,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,235,4.202128
1,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,219,4.214612
2,480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,218,3.786697
3,110,Braveheart (1995),Action|Drama|War,212,4.058962
4,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,206,4.264563
5,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi,200,3.970000
6,527,Schindler's List (1993),Drama|War,188,4.228723
7,780,Independence Day (a.k.a. ID4) (1996),Action|Adventure|Sci-Fi|Thriller,185,3.445946
8,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,184,3.923913
9,150,Apollo 13 (1995),Adventure|Drama|IMAX,184,3.875000


Already-rated movies were excluded.


In [15]:
# pip install scipy

In [16]:
from scipy.sparse import csr_matrix

user_ids = sorted(train_ratings["userId"].unique())
movie_ids = sorted(train_ratings["movieId"].unique())

user_to_index = {
    user_id: index for index, user_id in enumerate(user_ids)
}

movie_to_index = {
    movie_id: index for index, movie_id in enumerate(movie_ids)
}

user_rows = train_ratings["userId"].map(user_to_index).to_numpy()
movie_columns = train_ratings["movieId"].map(movie_to_index).to_numpy()

rating_matrix = csr_matrix(
    (
        train_ratings["rating"].to_numpy(),
        (user_rows, movie_columns)
    ),
    shape=(len(user_ids), len(movie_ids))
)

print("Matrix shape:", rating_matrix.shape)
print("Ratings stored:", rating_matrix.nnz)

Matrix shape: (522, 7867)
Ratings stored: 80668


In [17]:
# %pip install scikit-learn

In [18]:
from sklearn.neighbors import NearestNeighbors

movie_matrix = rating_matrix.T.tocsr()

model = NearestNeighbors(
    metric="cosine",
    algorithm="brute"
)

model.fit(movie_matrix)

print("Movie matrix shape:", movie_matrix.shape)

Movie matrix shape: (7867, 522)


In [19]:
movie_id = 1
movie_index = movie_to_index[movie_id]

distances, indices = model.kneighbors(
    movie_matrix[movie_index],
    n_neighbors=min(11, len(movie_ids))
)

similar_movies = pd.DataFrame({
    "movieId": [movie_ids[index] for index in indices[0]],
    "similarity": 1 - distances[0]
})

similar_movies = similar_movies[
    similar_movies["movieId"] != movie_id
]

In [20]:
similar_movies = similar_movies.merge(
    movies_data,
    on="movieId",
    how="left",
    validate="one_to_one"
)

selected_title = movies_data.loc[
    movies_data["movieId"] == movie_id, "title"
].iloc[0]

print("Movies with similar rating patterns to:", selected_title)

display(
    similar_movies[
        ["movieId", "title", "genres", "similarity"]
    ].head(10).reset_index(drop=True)
)

Movies with similar rating patterns to: Toy Story (1995)


,movieId,title,genres,similarity
0,780,Independence Day (a.k.a. ID4) (1996),Action|Adventure|Sci-Fi|Thriller,0.580841
1,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,0.565907
2,480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,0.560696
3,648,Mission: Impossible (1996),Action|Adventure|Mystery|Thriller,0.558224
4,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.552276
5,1210,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi,0.550037
6,356,Forrest Gump (1994),Comedy|Drama|Romance|War,0.545533
7,1265,Groundhog Day (1993),Comedy|Fantasy|Romance,0.535942
8,588,Aladdin (1992),Adventure|Animation|Children|Comedy|Musical,0.535576
9,1073,Willy Wonka & the Chocolate Factory (1971),Children|Comedy|Fantasy|Musical,0.534514


In [21]:
def search_similar_movies(movie_id, n=10):
    if movie_id not in movie_to_index:
        return pd.DataFrame(
            columns=["movieId", "title", "genres", "similarity"]
        )

    movie_index = movie_to_index[movie_id]

    distances, indices = model.kneighbors(
        movie_matrix[movie_index],
        n_neighbors=min(n + 1, len(movie_ids))
    )

    similar = pd.DataFrame({
        "movieId": [
            movie_ids[index] for index in indices[0]
        ],
        "similarity": 1 - distances[0]
    })

    similar = similar[
        similar["movieId"] != movie_id
    ].head(n)

    return similar.merge(
        movies_data,
        on="movieId",
        how="left",
        validate="one_to_one"
    )[
        ["movieId", "title", "genres", "similarity"]
    ]

In [22]:
find_similar = search_similar_movies(
    movie_id=356,
    n=10
)

display(find_similar)

,movieId,title,genres,similarity
0,480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,0.725067
1,318,"Shawshank Redemption, The (1994)",Crime|Drama,0.699974
2,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,0.686800
3,110,Braveheart (1995),Action|Drama|War,0.673821
4,150,Apollo 13 (1995),Adventure|Drama|IMAX,0.664792
5,500,Mrs. Doubtfire (1993),Comedy|Drama,0.650040
6,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi,0.630991
7,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,0.626829
8,590,Dances with Wolves (1990),Adventure|Drama|Western,0.617666
9,457,"Fugitive, The (1993)",Thriller,0.616108


In [23]:
user_ratings = {
    356: 5.0,   # Forrest Gump
    1: 4.5,     # Toy Story
    318: 5.0    # Shawshank Redemption
}

In [24]:
def recommend_for_user(user_ratings, n=10):
    scores = {}

    for movie_id, rating in user_ratings.items():
        if movie_id not in movie_to_index:
            continue

        rating_weight = rating - 3.0

        if rating_weight == 0:
            continue

        movie_index = movie_to_index[movie_id]

        distances, indices = model.kneighbors(
            movie_matrix[movie_index],
            n_neighbors=min(21, len(movie_ids))
        )

        for distance, index in zip(
            distances[0][1:],
            indices[0][1:]
        ):
            candidate_id = movie_ids[index]

            if candidate_id in user_ratings:
                continue

            similarity = 1 - distance

            scores[candidate_id] = (
                scores.get(candidate_id, 0)
                + similarity * rating_weight
            )

    recommendations = pd.DataFrame(
        scores.items(),
        columns=["movieId", "recommendation_score"]
    )

    recommendations = recommendations.sort_values(
        "recommendation_score",
        ascending=False
    ).head(n)

    return recommendations.merge(
        movies_data,
        on="movieId",
        how="left",
        validate="one_to_one"
    )[
        ["movieId", "title", "genres", "recommendation_score"]
    ]

In [25]:
user_ratings = {
    356: 5.0,
    1: 4.5,
    318: 5.0
}

personalized_recommendations = recommend_for_user(
    user_ratings,
    n=10
)

display(personalized_recommendations)

,movieId,title,genres,recommendation_score
0,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,3.562569
1,480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,3.398826
2,150,Apollo 13 (1995),Adventure|Drama|IMAX,3.353379
3,110,Braveheart (1995),Action|Drama|War,2.570387
4,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,2.543894
5,527,Schindler's List (1993),Drama|War,2.483700
6,50,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,2.403425
7,457,"Fugitive, The (1993)",Thriller,2.351811
8,590,Dances with Wolves (1990),Adventure|Drama|Western,2.330048
9,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi,2.328873


In [26]:
user_ratings = {
    356: 5.0,
    1: 4.5,
    318: 5.0,
    480: 1.0
}

personalized_recommendations = recommend_for_user(
    user_ratings,
    n=10
)

display(personalized_recommendations)

assert not personalized_recommendations["movieId"].isin(
    user_ratings.keys()
).any()

,movieId,title,genres,recommendation_score
0,527,Schindler's List (1993),Drama|War,2.483700
1,50,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,2.403425
2,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,2.331936
3,47,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,2.289748
4,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,2.284133
5,150,Apollo 13 (1995),Adventure|Drama|IMAX,2.064183
6,32,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller,1.828720
7,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,1.348528
8,597,Pretty Woman (1990),Comedy|Romance,1.218548
9,110,Braveheart (1995),Action|Drama|War,1.187178


In [27]:
def search_movies(query, n=10):
    query = query.strip()

    if not query:
        return pd.DataFrame(
            columns=["movieId", "title", "genres"]
        )

    matches = movies_data[
        movies_data["title"].str.contains(
            query,
            case=False,
            na=False,
            regex=False
        )
    ]

    return matches[
        ["movieId", "title", "genres"]
    ].head(n).reset_index(drop=True)

In [28]:
search_results = search_movies("Avenger", n=10)

display(search_results)

,movieId,title,genres
0,2153,"Avengers, The (1998)",Action|Adventure
1,3693,"Toxic Avenger, The (1985)",Comedy|Horror
2,3694,"Toxic Avenger, Part II, The (1989)",Comedy|Horror
3,3695,Toxic Avenger Part III: The Last Temptation of...,Comedy|Horror
4,44020,Ultimate Avengers (2006),Action|Animation|Children|Sci-Fi
5,88140,Captain America: The First Avenger (2011),Action|Adventure|Sci-Fi|Thriller|War
6,89745,"Avengers, The (2012)",Action|Adventure|Sci-Fi|IMAX
7,115727,Crippled Avengers (Can que) (Return of the 5 D...,Action|Adventure
8,122892,Avengers: Age of Ultron (2015),Action|Adventure|Sci-Fi
9,122912,Avengers: Infinity War - Part I (2018),Action|Adventure|Sci-Fi


In [29]:
def recommend_from_history(user_id, history, n=10):
    user_ratings = history[
        history["userId"] == user_id
    ].set_index("movieId")["rating"].to_dict()

    if not user_ratings:
        return []

    recommendations = recommend_for_user(
        user_ratings,
        n=n
    )

    return recommendations["movieId"].tolist()

In [30]:
warm_validation_users = sorted(
    set(train_ratings["userId"]) &
    set(validation_ratings["userId"])
)

user_id = warm_validation_users[0]

recommended_ids = recommend_from_history(
    user_id,
    train_ratings,
    n=10
)

recommended_movies = movies_data[
    movies_data["movieId"].isin(recommended_ids)
][["movieId", "title", "genres"]]

liked_in_validation = validation_ratings[
    (validation_ratings["userId"] == user_id) &
    (validation_ratings["rating"] >= 4.0)
].merge(
    movies_data,
    on="movieId",
    how="left"
)[["movieId", "title", "rating"]]

print("User:", user_id)
print("\nRecommended movies:")
display(recommended_movies)

print("\nMovies rated 4.0 or higher in validation:")
display(liked_in_validation)

User: 18

Recommended movies:


,movieId,title,genres
123,150,Apollo 13 (1995),Adventure|Drama|IMAX
398,457,"Fugitive, The (1993)",Thriller
509,592,Batman (1989),Action|Crime|Thriller
694,912,Casablanca (1942),Drama|Romance
902,1200,Aliens (1986),Action|Adventure|Horror|Sci-Fi
964,1265,Groundhog Day (1993),Comedy|Fantasy|Romance
1218,1617,L.A. Confidential (1997),Crime|Film-Noir|Mystery|Thriller
1979,2628,Star Wars: Episode I - The Phantom Menace (1999),Action|Adventure|Sci-Fi
2038,2716,Ghostbusters (a.k.a. Ghost Busters) (1984),Action|Comedy|Sci-Fi
2145,2858,American Beauty (1999),Drama|Romance



Movies rated 4.0 or higher in validation:


,movieId,title,rating
0,6807,Monty Python's The Meaning of Life (1983),4.5
1,5464,Road to Perdition (2002),4.0
2,1892,"Perfect Murder, A (1998)",4.0
3,55765,American Gangster (2007),4.5
4,1617,L.A. Confidential (1997),4.0
5,84392,"Lincoln Lawyer, The (2011)",4.5
6,6,Heat (1995),4.0
7,1221,"Godfather: Part II, The (1974)",5.0
8,745,Wallace & Gromit: A Close Shave (1995),4.0
9,16,Casino (1995),4.5


In [31]:
recommended_ids = set(recommended_movies["movieId"])

liked_ids = set(liked_in_validation["movieId"])

matched_ids = recommended_ids & liked_ids

matched_movies = movies_data[
    movies_data["movieId"].isin(matched_ids)
][["movieId", "title", "genres"]]

print("Recommended movies:", len(recommended_ids))
print("Movies rated 4.0 or higher later:", len(liked_ids))
print("Exact matches:", len(matched_ids))

display(matched_movies)

Recommended movies: 10
Movies rated 4.0 or higher later: 52
Exact matches: 1


,movieId,title,genres
1218,1617,L.A. Confidential (1997),Crime|Film-Noir|Mystery|Thriller
